# Full matched sweep — both arms, one protocol

Everything under a single protocol so I.P. and BaCP differ **only** in the objective. Prior records were archived to `runs_archive_preprotocol_20260823` (they ran 60 epochs, delta_T 88, and BaCP on 50,000 unsplit images).

| | dense | I.P. | BaCP |
|---|---|---|---|
| optimizer | SGD 0.01 | SGD 0.01 | SGD 0.1 (VGG 0.05) |
| epochs | 100, patience 20 | 50 | 50 + 25 fine-tune (AdamW 1e-4) |
| delta_T | — | 100 | 100 |
| val_split | 0.10 | 0.10 | 0.10 |
| objective | CE | CE | legacy 2Bx2B, proj_mode current, tau 0.15 |

Data: CIFAR-10, **45,000 train / 5,000 val / 10,000 held-out test**, batch 512 -> 87 batches/epoch. Cubic ramp over epochs 0-40, mask frozen for the last 10. Classifier pruned; projection head never pruned; `wanda_group=global`; bf16 + grad-clip 10.0 on both arms.

Grid: 4 models x 3 pruners x 4 sparsities x 2 arms, plus 4 dense = **100 cells, ~14 h**. Order is dense, then all I.P., then all BaCP — baselines land first, and within each arm the sweep is breadth-first (pruner -> sparsity -> model) so every pass completes one comparable row across all four models.

Safe to interrupt and re-run: a recorded cell is skipped.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check

Halts before any GPU time if the protocol is wrong.

In [ ]:
MODELS     = ['resnet34', 'vgg11', 'resnet50', 'vgg19']
PRUNERS    = ['magnitude', 'snip', 'wanda']
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEED, GPU  = 1, 0

plan  = [nb.make_cell(m, 'dense', seed=SEED) for m in MODELS]
for phase in ('prune', 'bacp'):            # baselines first, then the method
    for p in PRUNERS:
        for s in SPARSITIES:
            for m in MODELS:
                plan.append(nb.make_cell(m, phase, seed=SEED,
                                         pruner=p, sparsity=s))

print(f'{len(plan)} cells: {len(MODELS)} dense + '
      f'{len(MODELS)*len(PRUNERS)*len(SPARSITIES)} I.P. + '
      f'{len(MODELS)*len(PRUNERS)*len(SPARSITIES)} BaCP')
assert nb.sanity_check(plan), 'sanity check failed'

## Run

One row per epoch. `results.csv` is rewritten after every cell.

In [ ]:
for cell in plan:
    nb.run(cell, gpu=GPU)
    nb.update_results_csv()

## Table — re-run this cell any time

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if '.smoke' in k or r.get('status') != 'ok':
        continue
    acc[k] = r.get('test_acc_pct')

for m in MODELS:
    d = acc.get(f'static.dense.{m}.cifar10.dense.seed{SEED}')
    print(f'\n{m}   dense {d:.2f}' if d else f'\n{m}   dense -')
    print(f'  {"criterion":<11}' + ''.join(f'{s:>18}' for s in SPARSITIES))
    for p in PRUNERS:
        cells_ = []
        for s in SPARSITIES:
            i = acc.get(f'static.prune.{m}.cifar10.s{s}.{p}.seed{SEED}')
            b = acc.get(f'static.bacp.{m}.cifar10.s{s}.{p}.seed{SEED}')
            if i is None and b is None:
                cells_.append(f'{"-":>18}')
            else:
                istr = f'{i:.2f}' if i is not None else '-'
                bstr = f'{b:.2f}' if b is not None else '-'
                dstr = f'{b-i:+.2f}' if (i is not None and b is not None) else ''
                cells_.append(f'{istr}/{bstr}{dstr:>7}'.rjust(18))
        print(f'  {p:<11}' + ''.join(cells_))
print('\nformat: I.P./BaCP (delta)')